# URL classifier retraining

This notebook normalizes training and inference text consistently, removes conflicting labels, splits by hostname, evaluates on unseen hosts, and saves the pipeline only after evaluation.

In [3]:
from pathlib import Path
from urllib.parse import urlsplit
import pandas as pd
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DATA_PATH = Path("../data/malicious_phish.csv")
MODEL_PATH = Path("../models/tfidf_url_classifier.pkl")

In [4]:
def parse_host(url):
    value = str(url).strip().lower()
    candidate = value if "://" in value else "http://" + value
    try:
        return (urlsplit(candidate).hostname or "").lower().strip(".")
    except ValueError:
        return ""


def normalize_url(url):
    value = str(url).strip().lower()
    if not value:
        return ""
    candidate = value if "://" in value else "http://" + value
    try:
        parsed = urlsplit(candidate)
        host = (parsed.hostname or "").lower().strip(".")
        if not host:
            return value
        path = parsed.path.rstrip("/") if parsed.path in ("", "/") else parsed.path
        return host + path + ("?" + parsed.query if parsed.query else "")
    except ValueError:
        return value

In [5]:
df = pd.read_csv(DATA_PATH).dropna(subset=["url", "type"]).copy()
df["url"] = df["url"].astype(str).str.strip()
df["type"] = df["type"].astype(str).str.strip().str.lower()
df = df[df["url"] != ""]
df["model_text"] = df["url"].map(normalize_url)
df["host"] = df["url"].map(parse_host)
df = df[(df["model_text"] != "") & (df["host"] != "")]
df = df.drop_duplicates(subset=["model_text", "type"])
conflict_counts = df.groupby("model_text")["type"].nunique()
conflicts = conflict_counts[conflict_counts > 1].index
df = df[~df["model_text"].isin(conflicts)].reset_index(drop=True)
df = df.drop_duplicates(subset=["model_text"]).reset_index(drop=True)

print("Rows:", len(df))
print("Classes:")
print(df["type"].value_counts())
print("Conflicting normalized URLs removed:", len(conflicts))

Rows: 630928
Classes:
type
benign        423564
defacement     95262
phishing       89207
malware        22895
Name: count, dtype: int64
Conflicting normalized URLs removed: 4045


In [6]:
split1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(split1.split(df, df["type"], groups=df["host"]))
train = df.iloc[train_idx].copy()
temp = df.iloc[temp_idx].copy()

split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=43)
val_idx, test_idx = next(split2.split(temp, temp["type"], groups=temp["host"]))
val = temp.iloc[val_idx].copy()
test = temp.iloc[test_idx].copy()

train_hosts = set(train["host"])
val_hosts = set(val["host"])
test_hosts = set(test["host"])
print("Rows — train/validation/test:", len(train), len(val), len(test))
print("Host overlap — train/validation:", len(train_hosts & val_hosts))
print("Host overlap — train/test:", len(train_hosts & test_hosts))
print("Host overlap — validation/test:", len(val_hosts & test_hosts))
assert not (train_hosts & val_hosts or train_hosts & test_hosts or val_hosts & test_hosts), "Hostname leakage detected"

Rows — train/validation/test: 506058 66219 58651
Host overlap — train/validation: 0
Host overlap — train/test: 0
Host overlap — validation/test: 0


In [7]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        max_features=100000,
        sublinear_tf=True,
        dtype="float32"
    )),
    ("classifier", LogisticRegression(
        max_iter=1500,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(train["model_text"], train["type"])
val_pred = model.predict(val["model_text"])
print("Validation accuracy:", round(accuracy_score(val["type"], val_pred), 4))
print("Validation macro F1:", round(f1_score(val["type"], val_pred, average="macro"), 4))
print(classification_report(val["type"], val_pred, digits=4, zero_division=0))

/opt/anaconda3/envs/phishingml/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Validation accuracy: 0.884
Validation macro F1: 0.8755
              precision    recall  f1-score   support

      benign     0.9668    0.8796    0.9211     44233
  defacement     0.9560    0.8506    0.9003      8128
     malware     0.9268    0.9125    0.9196      1942
    phishing     0.6501    0.9182    0.7612     11916

    accuracy                         0.8840     66219
   macro avg     0.8749    0.8902    0.8755     66219
weighted avg     0.9073    0.8840    0.8898     66219



In [8]:

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

classes = list(model.classes_)
phishing_idx = classes.index("phishing")

val_probabilities = model.predict_proba(val["model_text"])
phishing_scores = val_probabilities[:, phishing_idx]
true_phishing = (val["type"] == "phishing").astype(int).to_numpy()

rows = []

for threshold in np.arange(0.30, 0.96, 0.05):
    predicted_phishing = (phishing_scores >= threshold).astype(int)

    rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(true_phishing, predicted_phishing, zero_division=0),
        "recall": recall_score(true_phishing, predicted_phishing, zero_division=0),
        "f1": f1_score(true_phishing, predicted_phishing, zero_division=0),
        "false_positive_rate": (
            ((predicted_phishing == 1) & (true_phishing == 0)).sum()
            / max(1, (true_phishing == 0).sum())
        )
    })

threshold_results = pd.DataFrame(rows)
display(threshold_results.round(4))


,threshold,precision,recall,f1,false_positive_rate
0,0.30,0.5556,0.9601,0.7038,0.1685
1,0.35,0.5852,0.9490,0.7239,0.1476
2,0.40,0.6131,0.9353,0.7407,0.1295
3,0.45,0.6409,0.9215,0.7560,0.1133
4,0.50,0.6683,0.9073,0.7696,0.0988
5,0.55,0.6936,0.8889,0.7792,0.0862
6,0.60,0.7199,0.8685,0.7872,0.0742
7,0.65,0.7488,0.8453,0.7941,0.0622
8,0.70,0.7837,0.8129,0.7980,0.0492
9,0.75,0.8250,0.7784,0.8010,0.0362


In [9]:
test_pred = model.predict(test["model_text"])
print("Held-out hostname test accuracy:", round(accuracy_score(test["type"], test_pred), 4))
print("Held-out hostname test macro F1:", round(f1_score(test["type"], test_pred, average="macro"), 4))
print(classification_report(test["type"], test_pred, digits=4, zero_division=0))
print("Confusion matrix labels:", list(model.classes_))
print(confusion_matrix(test["type"], test_pred, labels=model.classes_))

Held-out hostname test accuracy: 0.8881
Held-out hostname test macro F1: 0.8751
              precision    recall  f1-score   support

      benign     0.9544    0.8955    0.9240     35952
  defacement     0.9544    0.8494    0.8988      9738
     malware     0.9631    0.9374    0.9501      4536
    phishing     0.6227    0.8749    0.7276      8425

    accuracy                         0.8881     58651
   macro avg     0.8737    0.8893    0.8751     58651
weighted avg     0.9074    0.8881    0.8936     58651

Confusion matrix labels: ['benign', 'defacement', 'malware', 'phishing']
[[32195   229   114  3414]
 [  542  8271     3   922]
 [   66    88  4252   130]
 [  930    78    46  7371]]


In [10]:

import pandas as pd

test_results = test[["url", "type", "model_text"]].copy()
test_results["predicted_label"] = test_pred
test_results["phishing_probability"] = model.predict_proba(
    test_results["model_text"]
)[:, list(model.classes_).index("phishing")]

false_positives = test_results[
    (test_results["type"] == "benign")
    & (test_results["predicted_label"] == "phishing")
].sort_values("phishing_probability", ascending=False)

print("False positives:", len(false_positives))
display(false_positives[["url", "type", "predicted_label", "phishing_probability"]].head(30))


False positives: 3414


,url,type,predicted_label,phishing_probability
541132,www.edgarnet.com/,benign,phishing,0.998410
577889,www.acqi.info,benign,phishing,0.998143
571841,www.ajras_ds.ga2h.com/,benign,phishing,0.997032
544643,www.facebook.ncom.pk/,benign,phishing,0.996879
578612,www.tanphatcomputer.com/cgi/-/itaubankline.com.br,benign,phishing,0.996072
541277,www.msc-verify.de/,benign,phishing,0.995910
561638,www.dzyauipweutz.net,benign,phishing,0.995658
431557,homepages.ulb.ac.be/~pbouilla/,benign,phishing,0.993005
539636,www.fazetook.com/,benign,phishing,0.991700
213235,now.netapp.com/,benign,phishing,0.991566


In [11]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH, compress=3)
print("Saved:", MODEL_PATH.resolve())
print("Classes:", list(model.classes_))

Saved: /Users/sreevatsarajesh/Documents/GitHub/FML_project/NLP_classifier/models/tfidf_url_classifier.pkl
Classes: ['benign', 'defacement', 'malware', 'phishing']


In [12]:
test_urls = [
    "https://google.com",
    "https://github.com",
    "https://wikipedia.org",
    "https://example.com",
    "http://secure-login-example.com/verify-account"
]

for original in test_urls:
    text_value = normalize_url(original)
    probabilities = model.predict_proba([text_value])[0]
    scores = {label: round(float(score), 4) for label, score in zip(model.classes_, probabilities)}
    print(original, "=>", text_value, "=>", scores)

https://google.com => google.com => {'benign': 0.1678, 'defacement': 0.0001, 'malware': 0.068, 'phishing': 0.7641}
https://github.com => github.com => {'benign': 0.0111, 'defacement': 0.0001, 'malware': 0.0091, 'phishing': 0.9797}
https://wikipedia.org => wikipedia.org => {'benign': 0.8813, 'defacement': 0.0, 'malware': 0.0003, 'phishing': 0.1183}
https://example.com => example.com => {'benign': 0.0975, 'defacement': 0.0007, 'malware': 0.4193, 'phishing': 0.4825}
http://secure-login-example.com/verify-account => secure-login-example.com/verify-account => {'benign': 0.0291, 'defacement': 0.0002, 'malware': 0.0016, 'phishing': 0.9692}


In [13]:

from urllib.parse import urlsplit

def get_host(url):
    value = str(url).strip().lower()
    if "://" not in value:
        value = "http://" + value
    try:
        return urlsplit(value).hostname or ""
    except ValueError:
        return ""

false_positives["host"] = false_positives["url"].map(get_host)

print("Most common false-positive hosts:")
display(false_positives["host"].value_counts().head(20).rename("false_positive_count").to_frame())

host_label_counts = (
    df.assign(host=df["url"].map(get_host))
    .groupby("host")["type"]
    .nunique()
)

print("Hosts appearing with multiple labels:", int((host_label_counts > 1).sum()))


Most common false-positive hosts:


,false_positive_count
host,
9d345009-a-62cb3a1a-s-sites.googlegroups.com,148
archive.org,76
groups.yahoo.com,56
users6.nofeehost.com,30
www.cnhedge.cn,23
robtex.com,18
serviceinformation.pakfreewebs.com,16
www.twanvandewiel.nl,14
updateinformation.bankofamerica.com.ntamarine.com,13


Hosts appearing with multiple labels: 917


In [14]:

host_labels = (
    df.assign(host=df["url"].map(get_host))
    .groupby("host")
    .agg(
        labels=("type", lambda x: sorted(set(x))),
        label_count=("type", "nunique"),
        url_count=("url", "count")
    )
    .reset_index()
)

conflicting_hosts = host_labels[host_labels["label_count"] > 1]

print("Hosts with multiple labels:", len(conflicting_hosts))
display(conflicting_hosts.sort_values("url_count", ascending=False).head(30))


Hosts with multiple labels: 917


,host,labels,label_count,url_count
52838,en.wikipedia.org,"[benign, phishing]",2,13202
104097,myspace.com,"[benign, phishing]",2,3015
153004,twitter.com,"[benign, malware, phishing]",3,1641
113909,pastebin.com,"[benign, malware]",2,996
138495,sourceforge.net,"[benign, phishing]",2,971
61201,freepages.genealogy.rootsweb.ancestry.com,"[benign, phishing]",2,713
145046,techcrunch.com,"[benign, phishing]",2,448
47929,docs.google.com,"[benign, phishing]",2,434
96920,members.tripod.com,"[benign, phishing]",2,412
139455,sportsillustrated.cnn.com,"[benign, phishing]",2,365


In [15]:

fp_urls = set(false_positives["url"].astype(str))

label_check = (
    df[df["url"].astype(str).isin(fp_urls)]
    .groupby("url")["type"]
    .agg(lambda labels: sorted(set(labels)))
    .reset_index(name="dataset_labels")
)

print("False-positive URLs found in dataset:", len(label_check))
print("URLs with multiple labels:", int(
    label_check["dataset_labels"].map(len).gt(1).sum()
))

display(label_check.head(30))


False-positive URLs found in dataset: 3414
URLs with multiple labels: 0


,url,dataset_labels
0,'homeimprovementmessage.com/bahi\%20paypal/',[benign]
1,'institutocgr.com/web/administrator/help/dk-/S...,[benign]
2,'promosbr.net/xsl/?\%AC\%AC\%A3\%A2#sa\%20prem...,[benign]
3,'promosbr.net/xsl/?\%ac\%ac\%a3\%a2sa\%20premi...,[benign]
4,'recebepremios12.altervista.org/premios\%20300...,[benign]
5,"'survucus-rumescapa.3eeweb.com/forums.ws15,16,...",[benign]
6,'vpjawel.com/productssamples/products\%20purch...,[benign]
7,'www.casadavovo.com/flash/img/flash/bkp/temp/?...,[benign]
8,'www.clientebb.h19.ru/aspx/#\%20\%280.00784962...,[benign]
9,'www.dacristotal.com/~hailagra/signin/enter_lo...,[benign]


In [16]:

from sklearn.metrics import classification_report, confusion_matrix

test_probabilities = model.predict_proba(test["model_text"])
phishing_idx = list(model.classes_).index("phishing")
test_phishing_scores = test_probabilities[:, phishing_idx]
test_true = (test["type"] == "phishing").astype(int).to_numpy()

for threshold in [0.50, 0.75, 0.80]:
    test_pred = (test_phishing_scores >= threshold).astype(int)

    print(f"\nThreshold: {threshold:.2f}")
    print(classification_report(
        test_true,
        test_pred,
        labels=[0, 1],
        target_names=["not_phishing", "phishing"],
        digits=4,
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(test_true, test_pred, labels=[0, 1]))



Threshold: 0.50
              precision    recall  f1-score   support

not_phishing     0.9733    0.9208    0.9463     50226
    phishing     0.6427    0.8496    0.7318      8425

    accuracy                         0.9105     58651
   macro avg     0.8080    0.8852    0.8391     58651
weighted avg     0.9258    0.9105    0.9155     58651

Confusion matrix:
[[46246  3980]
 [ 1267  7158]]

Threshold: 0.75
              precision    recall  f1-score   support

not_phishing     0.9468    0.9700    0.9582     50226
    phishing     0.7903    0.6749    0.7280      8425

    accuracy                         0.9276     58651
   macro avg     0.8685    0.8224    0.8431     58651
weighted avg     0.9243    0.9276    0.9252     58651

Confusion matrix:
[[48717  1509]
 [ 2739  5686]]

Threshold: 0.80
              precision    recall  f1-score   support

not_phishing     0.9389    0.9776    0.9578     50226
    phishing     0.8229    0.6205    0.7075      8425

    accuracy                     

In [17]:

threshold = 0.80
test_pred = (test_phishing_scores >= threshold).astype(int)

print(classification_report(
    test_true,
    test_pred,
    labels=[0, 1],
    target_names=["not_phishing", "phishing"],
    digits=4,
    zero_division=0
))
print(confusion_matrix(test_true, test_pred, labels=[0, 1]))


              precision    recall  f1-score   support

not_phishing     0.9389    0.9776    0.9578     50226
    phishing     0.8229    0.6205    0.7075      8425

    accuracy                         0.9263     58651
   macro avg     0.8809    0.7991    0.8327     58651
weighted avg     0.9222    0.9263    0.9219     58651

[[49101  1125]
 [ 3197  5228]]


In [18]:

for threshold in [0.50, 0.75, 0.80]:
    pred = (test_phishing_scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        test_true, pred, labels=[0, 1]
    ).ravel()

    print(f"\nThreshold: {threshold:.2f}")
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"Predicted phishing: {pred.sum()}")
    print(f"Precision: {tp / max(1, tp + fp):.4f}")
    print(f"Recall: {tp / max(1, tp + fn):.4f}")



Threshold: 0.50
TN=46246, FP=3980, FN=1267, TP=7158
Predicted phishing: 11138
Precision: 0.6427
Recall: 0.8496

Threshold: 0.75
TN=48717, FP=1509, FN=2739, TP=5686
Predicted phishing: 7195
Precision: 0.7903
Recall: 0.6749

Threshold: 0.80
TN=49101, FP=1125, FN=3197, TP=5228
Predicted phishing: 6353
Precision: 0.8229
Recall: 0.6205


In [19]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix

baseline_model = model

def hostname_text(url):
    from urllib.parse import urlsplit

    value = str(url).strip().lower()
    candidate = value if "://" in value else "http://" + value

    try:
        parsed = urlsplit(candidate)
        host = (parsed.hostname or "").lower()
        path = parsed.path or "/"
        query = parsed.query
        return f"host {host} path {path} query {query}"
    except ValueError:
        return value

candidate_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        max_features=100000,
        sublinear_tf=True,
        dtype="float32"
    )),
    ("classifier", LogisticRegression(
        max_iter=1500,
        class_weight="balanced",
        random_state=42,
        C=0.5
    ))
])

candidate_train = train["model_text"].map(hostname_text)
candidate_val = val["model_text"].map(hostname_text)

candidate_model.fit(candidate_train, train["type"])

baseline_val_pred = baseline_model.predict(val["model_text"])
candidate_val_pred = candidate_model.predict(candidate_val)

print("Baseline validation macro F1:",
      round(f1_score(val["type"], baseline_val_pred, average="macro"), 4))
print("Candidate validation macro F1:",
      round(f1_score(val["type"], candidate_val_pred, average="macro"), 4))

print("\nBaseline validation report")
print(classification_report(
    val["type"], baseline_val_pred, digits=4, zero_division=0
))

print("\nCandidate validation report")
print(classification_report(
    val["type"], candidate_val_pred, digits=4, zero_division=0
))


/opt/anaconda3/envs/phishingml/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Baseline validation macro F1: 0.8755
Candidate validation macro F1: 0.8783

Baseline validation report
              precision    recall  f1-score   support

      benign     0.9668    0.8796    0.9211     44233
  defacement     0.9560    0.8506    0.9003      8128
     malware     0.9268    0.9125    0.9196      1942
    phishing     0.6501    0.9182    0.7612     11916

    accuracy                         0.8840     66219
   macro avg     0.8749    0.8902    0.8755     66219
weighted avg     0.9073    0.8840    0.8898     66219


Candidate validation report
              precision    recall  f1-score   support

      benign     0.9639    0.8737    0.9166     44233
  defacement     0.9496    0.8859    0.9167      8128
     malware     0.9266    0.9171    0.9218      1942
    phishing     0.6507    0.9075    0.7580     11916

    accuracy                         0.8826     66219
   macro avg     0.8727    0.8961    0.8783     66219
weighted avg     0.9047    0.8826    0.8882     66219

In [20]:

from sklearn.base import clone
from sklearn.metrics import classification_report, f1_score
from sklearn.linear_model import LogisticRegression

regularization_model = clone(baseline_model)
regularization_model.set_params(classifier__C=0.5)
regularization_model.fit(train["model_text"], train["type"])

regularization_pred = regularization_model.predict(val["model_text"])

print("Baseline macro F1:",
      round(f1_score(val["type"], baseline_model.predict(val["model_text"]), average="macro"), 4))

print("C=0.5 macro F1:",
      round(f1_score(val["type"], regularization_pred, average="macro"), 4))

print(classification_report(
    val["type"],
    regularization_pred,
    digits=4,
    zero_division=0
))


/opt/anaconda3/envs/phishingml/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Baseline macro F1: 0.8755
C=0.5 macro F1: 0.8736
              precision    recall  f1-score   support

      benign     0.9673    0.8751    0.9189     44233
  defacement     0.9523    0.8530    0.8999      8128
     malware     0.9207    0.9145    0.9176      1942
    phishing     0.6447    0.9194    0.7579     11916

    accuracy                         0.8815     66219
   macro avg     0.8713    0.8905    0.8736     66219
weighted avg     0.9060    0.8815    0.8876     66219



In [21]:

from sklearn.base import clone
from sklearn.metrics import classification_report, f1_score

hostname_model = clone(candidate_model)
hostname_model.set_params(classifier__C=1.0)
hostname_model.fit(candidate_train, train["type"])

baseline_pred = baseline_model.predict(val["model_text"])
hostname_pred = hostname_model.predict(candidate_val)

print("Baseline macro F1:",
      round(f1_score(val["type"], baseline_pred, average="macro"), 4))
print("Hostname representation macro F1:",
      round(f1_score(val["type"], hostname_pred, average="macro"), 4))

print("\nHostname representation report")
print(classification_report(
    val["type"], hostname_pred, digits=4, zero_division=0
))


/opt/anaconda3/envs/phishingml/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Baseline macro F1: 0.8755
Hostname representation macro F1: 0.874

Hostname representation report
              precision    recall  f1-score   support

      benign     0.9531    0.8787    0.9144     44233
  defacement     0.9519    0.8781    0.9135      8128
     malware     0.9331    0.9192    0.9261      1942
    phishing     0.6469    0.8700    0.7420     11916

    accuracy                         0.8783     66219
   macro avg     0.8712    0.8865    0.8740     66219
weighted avg     0.8972    0.8783    0.8836     66219

